# Reverse-QTY series C MD, part 4: where the extra mobility sits

Taner Karagol and Alper Karagol.

Part 3 returned a verdict on the interface. It also showed that the converted build
has the higher backbone RMSD of the two while its interface is intact, which says
the extra motion is somewhere other than the dimer contact. This notebook finds
where, by measuring the per-residue fluctuation of both builds over the same
equilibrated window and apportioning the difference between the chain ends, the
interface, the converted positions and the rest.

**No GPU is needed.** A CPU runtime is enough and spends no accelerator units. Set
the runtime to CPU, then Runtime > Run all.

It reads the trajectories part 2 already staged to Drive, so nothing is simulated
again. Both runs must have finished and part 2 must have written their reports; the
folder each run used is taken from its own report rather than rebuilt from a path
here, so a run that lived somewhere unusual is still found.

It writes `md/results/rmsf.json` and `figures/rmsf.png` and `.svg`.


In [ ]:
#@title 1. Settings
PROJECT_DIR     = "" #@param {type:"string"}
#@markdown The project folder, used when it is not where DRIVE_ROOT and
#@markdown PROJECT_FOLDER below point. Set it to wherever you put this folder,
#@markdown on Drive or on the Colab disk, and every other path follows from it.
#@markdown Left empty it falls back to those two.
DRIVE_ROOT      = "/content/drive/MyDrive"  #@param {type:"string"}
PROJECT_FOLDER  = "14-3-3-rQTY" #@param {type:"string"}
TERMINAL_RESIDUES = 10   #@param {type:"integer"}
#@markdown Residues at each chain end counted as terminal. The tails of this protein
#@markdown are mobile in both builds, so they are reported apart from the body
#@markdown instead of being averaged into it.
TOP_N           = 12     #@param {type:"integer"}
#@markdown How many residues to list by their increase in fluctuation.
SELECTION       = "C-alpha" #@param ["C-alpha", "Protein"]
#@markdown What the fluctuation is measured on. C-alpha gives one value per residue
#@markdown and is the backbone counterpart of the RMSD part 3 reports.
GMX_VER         = "2026.3" #@param {type:"string"}
#@markdown Only used to find the cached build on Drive. If that binary will not run
#@markdown without a GPU, a packaged GROMACS is installed instead and the result
#@markdown records which one measured it.

import os
PROJECT = PROJECT_DIR.strip() or os.path.join(DRIVE_ROOT, PROJECT_FOLDER)
MD = os.path.join(PROJECT, "md")
RESULTS = os.path.join(MD, "results")
FIGURES = os.path.join(PROJECT, "figures")
DATA = os.path.join(PROJECT, "data")
INPUTS = os.path.join(MD, "inputs")
WORK = "/content/work4"
BUILD_ORDER = ["native", "converted"]
print("project    ", PROJECT)
print("measuring   per-residue fluctuation on %s, over the window part 2 kept"
      % SELECTION)


In [ ]:
#@title 2. Mount Drive, get GROMACS, measure the fluctuation
from google.colab import drive
drive.mount('/content/drive')

import json, glob, re, shutil, subprocess, datetime
import numpy as np

PREFIX = "/usr/local/gromacs"
os.environ["PATH"] = PREFIX + "/bin:" + os.environ["PATH"]


def fatal(msg):
    raise SystemExit("STOP: " + msg)


def sh(cmd, label="", cwd=None):
    r = subprocess.run(cmd, shell=True, text=True, cwd=cwd,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        print(r.stdout[-2000:])
        fatal("%s failed" % (label or cmd))
    return r.stdout


def gmx_version():
    try:
        r = subprocess.run(["gmx", "--version"], text=True,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        return r.stdout if r.returncode == 0 else ""
    except Exception:
        return ""


def gmx(*args, stdin_text=None, label=""):
    cmd = ["gmx"] + [str(a) for a in args]
    print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, text=True, input=stdin_text,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        print(r.stdout[-3000:])
        fatal("%s failed (exit %d). The output above is the reason."
              % (label or cmd[1], r.returncode))
    return r.stdout


def calpha_rows(pdb_path, selection):
    """-> list of (chain_or_segid, resid, resname), in file order.

    Indexed by position in the file. The serial column of a deposited file is not
    contiguous, so it is not used to order anything.
    """
    want_ca = selection == "C-alpha"
    out = []
    for L in open(pdb_path, errors="replace"):
        if not L.startswith(("ATOM", "HETATM")):
            continue
        name = L[12:16].strip()
        if want_ca and name != "CA":
            continue
        seg = L[72:76].strip() or L[21].strip() or "?"
        out.append((seg, int(L[22:26]), L[17:20].strip()))
    return out


def read_xvg(path):
    """-> (x, y, yaxis label). The comment lines are skipped, except the one
    that names the y axis, which is kept because it carries the unit.
    """
    xs, ys, unit = [], [], None
    for L in open(path, errors="replace"):
        s = L.strip()
        if s.startswith("@") and "yaxis" in s and "label" in s:
            m = re.search(r'"(.*)"', s)
            if m:
                unit = m.group(1)
            continue
        if not s or s[0] in "#@":
            continue
        p = s.split()
        if len(p) < 2:
            continue
        xs.append(float(p[0]))
        ys.append(float(p[1]))
    return np.array(xs), np.array(ys), unit


def to_angstrom(y, unit, path):
    """A length column in the unit the file itself declares, as angstrom.

    GROMACS works in nanometres and writes them unconverted, so a column taken
    at face value is wrong by a factor of ten. This reads the unit off the file
    rather than assuming it, and refuses a unit it cannot name instead of
    guessing, so a change in what GROMACS writes stops the notebook.
    """
    u = (unit or "").lower()
    if "nm" in u or "nanomet" in u:
        return y * 10.0, "nm"
    if "angstrom" in u or "(a)" in u or "\u00c5" in (unit or ""):
        return y, "angstrom"
    fatal("%s declares its y axis as %r. Neither nanometre nor angstrom is in "
          "that, so the unit of these numbers is unknown and calling them "
          "angstrom could be wrong by a factor of ten."
          % (os.path.basename(path), unit))


def burn_ps(rep):
    """The window part 2 kept, in picoseconds, taken from its own report."""
    if rep["frames"] < 2:
        fatal("the %s report holds %d frames" % (rep["build"], rep["frames"]))
    return rep["burn"] / float(rep["frames"] - 1) * rep["ns"] * 1000.0


def apportion(keys, values, iface, converted, n_term, chains):
    """Mean fluctuation over each class of residue, and the counts behind it.

    terminal and body split the chain by where a residue sits. interface and
    converted are overlays on that split and are not slices of it, so a converted
    interface position lying near a chain end is counted in all three and the four
    counts do not sum to the number of residues.
    """
    ends = set()
    for ch in chains:
        rs = sorted(r for c, r in keys if c == ch)
        ends.update((ch, r) for r in rs[:n_term] + rs[-n_term:])
    classes = {"terminal": lambda k: k in ends,
               "interface": lambda k: k[1] in iface,
               "converted": lambda k: k[1] in converted,
               "body": lambda k: (k not in ends and k[1] not in iface
                                  and k[1] not in converted)}
    out = {}
    for name, test in classes.items():
        sel = [values[k] for k in keys if test(k)]
        out[name] = dict(n=len(sel),
                         mean=float(np.mean(sel)) if sel else None)
    return out


# ---------------------------------------------------------------- the two runs
REP = {}
for b in BUILD_ORDER:
    p = os.path.join(RESULTS, "%s_report.json" % b)
    if not os.path.isfile(p):
        fatal("no %s. Run part 2 for the %s build first." % (p, b))
    REP[b] = json.load(open(p))

ns = {b: REP[b]["ns"] for b in BUILD_ORDER}
if abs(ns["native"] - ns["converted"]) > 0.05 * max(ns.values()):
    fatal("the two runs span %s ns. Comparing their fluctuation needs matched "
          "sampling, so let the shorter one catch up first." % ns)
if len({REP[b]["frames"] for b in BUILD_ORDER}) != 1:
    fatal("the two runs hold %s frames. The window kept would not be the same "
          "length." % {b: REP[b]["frames"] for b in BUILD_ORDER})

RUN = {}
for b in BUILD_ORDER:
    base = REP[b].get("run")
    if not base:
        fatal("the %s report does not say which folder it ran from, so it was "
              "written by an earlier part 2. Re-run part 2 for that build." % b)
    for f in ("prod_fit.xtc", "prod_ref.pdb"):
        if not os.path.isfile(os.path.join(base, f)):
            fatal("no %s in %s. Part 2 stages it at the end of a finished run, so "
                  "that run did not reach its measurement step."
                  % (f, base))
    RUN[b] = base
    print("%-10s %s" % (b, base))

SPEC = json.load(open(os.path.join(INPUTS, "build_spec.json")))
CONVERTED = set(SPEC["positions"])
ifj = json.load(open(os.path.join(DATA, "iface_1433.json")))
rec = [s for s in ifj["structures"] if s["pdb"] == SPEC["pdb_id"]]
if len(rec) != 1:
    fatal("%d records for %s in iface_1433.json" % (len(rec), SPEC["pdb_id"]))
IFACE = set(rec[0]["iface"])
print("%d converted positions, %d measured interface residues"
      % (len(CONVERTED), len(IFACE)))

# ------------------------------------------------------------------- gromacs
src = None
if "GROMACS" in gmx_version():
    src = "already in this session"
else:
    cache = os.path.join(PROJECT, "gromacs_cuda_%s.tar.gz" % GMX_VER)
    if os.path.isfile(cache):
        print("\nrestoring the cached GROMACS from", cache)
        try:
            sh("tar xzf '%s' -C /" % cache, label="restore cache")
        except SystemExit:
            shutil.rmtree(PREFIX, ignore_errors=True)
        if "GROMACS" in gmx_version():
            src = "cached build %s from the project folder" % GMX_VER
        else:
            print("the cached build does not run here, which is expected on a "
                  "runtime with no GPU")
            shutil.rmtree(PREFIX, ignore_errors=True)
if src is None:
    print("\ninstalling a packaged GROMACS, which is enough for this measurement")
    sh("apt-get -qq update && apt-get -qq install -y gromacs", label="apt")
    if "GROMACS" in gmx_version():
        src = "packaged gromacs from apt"
if src is None:
    fatal("no gmx on this runtime and neither the cached build nor the package "
          "would install.")
print("gromacs:", src)

# ------------------------------------------------------------- the measurement
os.makedirs(WORK, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
RMSF, KEYS, RESNAME, GMXRES = {}, {}, {}, {}
for b in BUILD_ORDER:
    b0 = burn_ps(REP[b])
    out = os.path.join(WORK, "rmsf_%s.xvg" % b)
    gmx("rmsf", "-s", os.path.join(RUN[b], "prod_ref.pdb"),
        "-f", os.path.join(RUN[b], "prod_fit.xtc"), "-o", out, "-res",
        "-b", "%g" % b0, stdin_text=SELECTION + "\n", label="rmsf " + b)
    if not os.path.isfile(out):
        fatal("gmx rmsf reported success but %s was not written" % out)
    rows = calpha_rows(os.path.join(RUN[b], "prod_ref.pdb"), SELECTION)
    x, y, yunit = read_xvg(out)
    y, UNIT_READ = to_angstrom(y, yunit, out)
    if len(y) != len(rows):
        fatal("%s holds %d values and prod_ref.pdb holds %d %s atoms. Pairing them "
              "by position would mislabel every residue."
              % (os.path.basename(out), len(y), len(rows), SELECTION))
    KEYS[b] = [(c, r) for c, r, _ in rows]
    RMSF[b] = {k: float(v) for k, v in zip(KEYS[b], y)}
    RESNAME[b] = {(c, r): nm for c, r, nm in rows}
    GMXRES[b] = {k: int(round(v)) for k, v in zip(KEYS[b], x)}
    print("  %-10s %d residues, skipped the first %.0f ps as part 2 did, "
          "%s read as %s"
          % (b, len(rows), b0, os.path.basename(out), UNIT_READ))
    shutil.copy2(out, os.path.join(RUN[b], "rmsf_%s.xvg" % b))

if KEYS["native"] != KEYS["converted"]:
    a, c = set(KEYS["native"]), set(KEYS["converted"])
    fatal("the two builds do not cover the same residues: %d only in native, %d "
          "only in converted. They are not the same system."
          % (len(a - c), len(c - a)))

keys = KEYS["native"]
chains = sorted({c for c, _ in keys})
diff = {k: RMSF["converted"][k] - RMSF["native"][k] for k in keys}
top = sorted(keys, key=lambda k: -diff[k])[:TOP_N]

NORM = {}
for b in BUILD_ORDER:
    v = [RMSF[b][k] for k in keys]
    lo, span = min(v), max(v) - min(v)
    NORM[b] = {k: ((RMSF[b][k] - lo) / span if span else 0.0) for k in keys}


def summarise(b):
    """The run and the spread of its fluctuation, per chain."""
    rep, per = REP[b], {}
    for ch in chains:
        v = [RMSF[b][k] for k in keys if k[0] == ch]
        rs = [r for c, r in keys if c == ch]
        per[ch] = dict(residues=len(v), first_residue=min(rs),
                       last_residue=max(rs), rmsf_A_mean=float(np.mean(v)),
                       rmsf_A_min=float(np.min(v)), rmsf_A_max=float(np.max(v)))
    return dict(ns=rep["ns"], frames_total=rep["frames"],
                frames_scored=rep["frames"] - rep["burn"],
                burn_ps=burn_ps(rep),
                spacing_ps=rep["ns"] * 1000.0 / (rep["frames"] - 1),
                chains=per)


OUT = dict(
    written=datetime.datetime.now().isoformat(timespec="seconds"),
    measured_with=src, selection=SELECTION,
    units=("angstrom. gmx writes the xvg in %s and it is converted on the way "
           "in; the xvg files kept beside each run are gmx output, unconverted"
           % UNIT_READ),
    ns={b: REP[b]["ns"] for b in BUILD_ORDER},
    frames={b: REP[b]["frames"] for b in BUILD_ORDER},
    window_skipped_ps={b: burn_ps(REP[b]) for b in BUILD_ORDER},
    chains=chains, residues=len(keys),
    summary={b: summarise(b) for b in BUILD_ORDER},
    class_note=("terminal and body are a disjoint split by position; interface and "
                "converted are overlays on it, so the four counts do not sum to the "
                "residue total"),
    terminal_residues_per_end=TERMINAL_RESIDUES,
    mean_A={b: float(np.mean([RMSF[b][k] for k in keys])) for b in BUILD_ORDER},
    by_class={b: apportion(keys, RMSF[b], IFACE, CONVERTED, TERMINAL_RESIDUES,
                           chains) for b in BUILD_ORDER},
    largest_increases=[dict(chain=c, residue=r, native_A=RMSF["native"][(c, r)],
                            converted_A=RMSF["converted"][(c, r)],
                            change_A=diff[(c, r)],
                            converted_position=r in CONVERTED,
                            interface=r in IFACE) for c, r in top],
    per_residue={b: [dict(chain=c, residue=r, rmsf_A=RMSF[b][(c, r)])
                     for c, r in keys] for b in BUILD_ORDER})

WROTE = [os.path.join(RESULTS, "rmsf.json")]
json.dump(OUT, open(WROTE[0], "w"), indent=1)

for b in BUILD_ORDER:
    for ch in chains:
        p = os.path.join(RESULTS, "rmsf_%s_%s.tsv" % (b, ch))
        with open(p, "w") as fh:
            fh.write("residue\tresname\tgmx_residue\trmsf_A\n")
            for c, r in keys:
                if c == ch:
                    fh.write("%d\t%s\t%d\t%.4f\n"
                             % (r, RESNAME[b][(c, r)], GMXRES[b][(c, r)],
                                RMSF[b][(c, r)]))
        WROTE.append(p)

p = os.path.join(RESULTS, "rmsf_table.tsv")
with open(p, "w") as fh:
    fh.write("\t".join(["chain", "residue", "resname_native",
                        "resname_converted", "rmsf_native_A",
                        "rmsf_converted_A", "delta_A", "norm_dynamics_native",
                        "norm_dynamics_converted", "converted_position",
                        "interface", "terminal"]) + "\n")
    ends = set()
    for ch in chains:
        rs = sorted(r for c, r in keys if c == ch)
        ends.update((ch, r) for r in rs[:TERMINAL_RESIDUES]
                    + rs[-TERMINAL_RESIDUES:])
    for k in keys:
        c, r = k
        na = round(RMSF["native"][k], 4)
        co = round(RMSF["converted"][k], 4)
        fh.write("%s\t%d\t%s\t%s\t%.4f\t%.4f\t%+.4f\t%.4f\t%.4f"
                 "\t%d\t%d\t%d\n"
                 % (c, r, RESNAME["native"][k], RESNAME["converted"][k],
                    na, co, co - na,
                    NORM["native"][k], NORM["converted"][k],
                    int(r in CONVERTED), int(r in IFACE), int(k in ends)))
WROTE.append(p)
print("\nmean fluctuation, %s" % SELECTION)
for b in BUILD_ORDER:
    print("  %-10s %.2f A overall" % (b, OUT["mean_A"][b]))
print("\nby class, native against converted. terminal and body split the chain by")
print("position; interface and converted overlay that split, so the counts below do")
print("not sum to the %d residues measured." % len(keys))
for cls in ("terminal", "interface", "converted", "body"):
    a = OUT["by_class"]["native"][cls]
    c = OUT["by_class"]["converted"][cls]
    if a["mean"] is None:
        continue
    print("  %-10s n=%3d   %.2f -> %.2f A   %+.2f"
          % (cls, a["n"], a["mean"], c["mean"], c["mean"] - a["mean"]))
print("\nlargest increases")
for e in OUT["largest_increases"]:
    tag = ",".join([t for t, on in (("converted", e["converted_position"]),
                                    ("interface", e["interface"])) if on]) or "-"
    print("  %s%-4d  %.2f -> %.2f A  %+.2f   %s"
          % (e["chain"], e["residue"], e["native_A"], e["converted_A"],
             e["change_A"], tag))
print("\nwrote")
for p in WROTE:
    print("  " + os.path.relpath(p, PROJECT))


In [ ]:
#@title 3. The figure
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

COL = {"native": "#4C6EF5", "converted": "#E8590C"}
R = json.load(open(os.path.join(RESULTS, "rmsf.json")))
os.makedirs(FIGURES, exist_ok=True)

fig, ax = plt.subplots(len(R["chains"]), 1, figsize=(11.0, 3.2 * len(R["chains"])),
                       facecolor="white", squeeze=False)
for a, ch in zip(ax[:, 0], R["chains"]):
    for b in BUILD_ORDER:
        rows = [e for e in R["per_residue"][b] if e["chain"] == ch]
        a.plot([e["residue"] for e in rows], [e["rmsf_A"] for e in rows],
               lw=1.0, color=COL[b], label=b)
    rows = [e for e in R["per_residue"]["native"] if e["chain"] == ch]
    top = max(max(e["rmsf_A"] for e in rows),
              max(e["rmsf_A"] for e in R["per_residue"]["converted"]
                  if e["chain"] == ch))
    for r in sorted(CONVERTED):
        if any(e["residue"] == r for e in rows):
            a.plot([r], [top * 1.04], marker="v", ms=4, color="#868E96",
                   clip_on=False)
    a.set_xlabel("residue")
    a.set_ylabel("C$\\alpha$ fluctuation, $\\mathrm{\\AA}$")
    a.set_title("chain %s" % ch, fontsize=10, loc="left")
    a.set_facecolor("white")
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
ax[0, 0].legend(frameon=False, fontsize=9, loc="upper center", ncol=2)
ax[0, 0].annotate("markers above the trace are the converted positions",
                  xy=(0.0, 1.14), xycoords="axes fraction", fontsize=8,
                  color="#868E96")
fig.suptitle("Reverse-QTY series C: per-residue fluctuation over %g ns, %s"
             % (R["ns"]["native"], R["selection"]), y=0.995)
fig.tight_layout()
for ext in ("png", "svg"):
    p = os.path.join(FIGURES, "rmsf.%s" % ext)
    fig.savefig(p, dpi=400 if ext == "png" else None, facecolor="white")
    print("wrote", p)


## Done

`md/results/rmsf.json` holds the per-residue fluctuation of both builds, their means
by class of residue, the residues whose fluctuation rose most, and a per-chain summary
of each run. `figures/rmsf.png` and `.svg` plot the two traces per chain with the
converted positions marked.

Beside the JSON it writes plain tables. `rmsf_<build>_<chain>.tsv` carries one chain of
one build, with the residue number, the residue name as that build has it, the number
GROMACS used, and the fluctuation. `rmsf_table.tsv` puts both builds on one residue
index, with the difference between them, each build's fluctuation rescaled to the range
that build covers, and flags for the converted positions, the measured interface and
the chain ends. The rescaled column is the normalisation step of the dynamic conserved
score, so a per-residue conservation column joined onto this table is all that score
needs; it is taken over the whole dimer so the two subunits stay on one scale.

Every number reported here is in angstrom. GROMACS writes the xvg in nanometres, and
the notebook converts by the unit the file declares rather than by assumption; the
`rmsf_*.xvg` kept beside each run are gmx output and are still in nanometres.

This measures where the two builds differ in mobility. It does not say why, and with
one trajectory per build the difference at any single residue is not separable from
the spread a second run would show.
